# image manipulation in python
a notebook to learn a few basics about image manipulation

In [ ]:
pip install readlif

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

next we will load the packages that we need

In [ ]:
from readlif.reader import LifFile
from matplotlib import pyplot as plt
import numpy as np
from skimage.filters import gaussian

In [ ]:
import os
# Clone the repo if not already in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/NBCimageAnalysis'):
        !git clone https://github.com/FilLieb/NBCimageAnalysis.git
    os.chdir('/content/NBCimageAnalysis/learning/')
    print(os.listdir('.'))

## load and display an image
next we will load an image, i.e. an .lif file (Leica's file format) and directly convert to a stack of arrays

In [ ]:
lif = LifFile('../data/2026group2/Gruppe2 WT - DMet High Density.lif')
image = lif.get_image(0)
channels_array = np.stack([np.array(channel) for channel in image.get_iter_c(t=0, z=0)])
print(channels_array.shape)  # (channels, height, width)

after that let's display the image to check everything is there

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 4))

axes[0].imshow(channels_array[0], cmap='gray')
axes[0].set_title("DAAO")

axes[1].imshow(channels_array[1], cmap='gray')
axes[1].set_title("vGAT")

axes[2].imshow(channels_array[2], cmap='gray')
axes[2].set_title("Cre")

axes[3].imshow(channels_array[3], cmap='gray')
axes[3].set_title("Gephyrin")

plt.tight_layout()
plt.show()

as you can see x- and y- coordinates are displayed in pixel numbers, we can retrieve the actual dimensions from the image's metadata

In [ ]:
x_scale, y_scale, z_scale, t_scale = image.scale

print(f"pixels per µm in x: {x_scale}")
print(f"pixels per µm in y: {y_scale}")
print(f"pixels per µm in z: {z_scale}")


since we are working with a single image plane, z has no meaning and we can convert the size of a pixel only in x and y:

In [ ]:
x_pixels = 1 / x_scale
y_pixels = 1 / y_scale


print(f"pixel size in x: {x_pixels} µm")
print(f"pixel size in y: {y_pixels} µm")

## basic image transformations
let's learn some basic image transformations, we will be using image filters scikit-image

to start with, we will be working with a single channel for simplicity

In [ ]:
gphn = channels_array[3]
gphn.shape

In [ ]:
plt.imshow(gphn, cmap='gray')

In [ ]:
plt.hist(gphn.ravel(), bins=256)
plt.show()

next we will use a Gaussian filter and display the resulting image in order to smooth the image

In [ ]:
high = gaussian(gphn, sigma=10, preserve_range=True)
plt.imshow(high, cmap='gray')

in this smooth image, gephyrin clusters cannot be seen, finally we will apply two gaussian filters (separately) and then subtract these images (pixel by pixel) from one another, this is called difference of Gaussians (DoG) and this method can be used to enhance certain features of an image

In [ ]:
plt.hist(high.ravel(), bins=256)
plt.show()

In [ ]:
def sub_to_zero(a, b):
    # Element-wise subtraction and maximum with zero
    return np.maximum(a - b, 0)


# Process Gphn channel (Difference of Gaussian)
low = gaussian(gphn, sigma=2, preserve_range=True)
high = gaussian(gphn, sigma=10, preserve_range=True)

dog = sub_to_zero(low,high)

# make a quick figure for display
fig, axes = plt.subplots(1, 4, figsize=(12, 4))

axes[0].imshow(gphn, cmap='gray')
axes[0].set_title("original")

axes[1].imshow(low, cmap='gray')
axes[1].set_title("low gaussian")

axes[2].imshow(high, cmap='gray')
axes[2].set_title("high gaussian")

axes[3].imshow(dog, cmap='gray')
axes[3].set_title("difference")

plt.tight_layout()
plt.show()

In [ ]:
# make a quick figure to display individual channels
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(gphn, cmap='gray')
axes[0].set_title("original")

axes[1].imshow(dog, cmap='gray')
axes[1].set_title("difference")

plt.tight_layout()
plt.show()
